In [18]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import random
import time

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

import timm

In [19]:
candidates = [
    Path('train_data'),
    Path('../train_data'),
    Path.cwd() / 'train_data',
    Path.cwd().parent / 'train_data',
]
train_data_dir = next((p.resolve() for p in candidates if p.exists()), None)
if train_data_dir is None:
    raise FileNotFoundError(
        "Could not find 'train_data' folder. Tried: " + ", ".join(str(p) for p in candidates)
    )
print('Using train_data:', train_data_dir)

seed = 42
TRAIN_FRAC = 0.90  # train uses only 90% of images

# Pretrained ImageNet backbones (including DenseNet121) expect ImageNet normalization
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

dataset = datasets.ImageFolder(root=str(train_data_dir), transform=transform)
num_classes = len(dataset.classes)
print('Num classes:', num_classes)

# Deterministic 90/10 split (disjoint)
n_all = len(dataset)
k_train = int(round(TRAIN_FRAC * n_all))
k_train = max(1, min(k_train, n_all - 1))  # ensure at least 1 train and 1 test

g = torch.Generator().manual_seed(seed)
perm = torch.randperm(n_all, generator=g)
train_idx = perm[:k_train].tolist()
test_idx = perm[k_train:].tolist()

train_img = Subset(dataset, train_idx)
test_img = Subset(dataset, test_idx)

batch_size = 32
num_workers = 4

g_loader = torch.Generator().manual_seed(seed)
train_loader = DataLoader(
    train_img,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    generator=g_loader,
 )
test_loader = DataLoader(
    test_img,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    generator=g_loader,
 )

print('Total images:', n_all)
print(f"Train images: {len(train_img)} ({len(train_img)/n_all:.0%})")
print(f"Test images:  {len(test_img)} ({len(test_img)/n_all:.0%})")
print('Classes (folder order used for labels):')
for i, name in enumerate(dataset.classes):
    print(f'  {i}: {name}')


Using train_data: /content/train_data
Num classes: 30
Total images: 6993
Train images: 6294 (90%)
Test images:  699 (10%)
Classes (folder order used for labels):
  0: Airport
  1: BareLand
  2: BaseballField
  3: Beach
  4: Bridge
  5: Center
  6: Church
  7: Commercial
  8: DenseResidential
  9: Desert
  10: Farmland
  11: Forest
  12: Industrial
  13: Meadow
  14: MediumResidential
  15: Mountain
  16: Park
  17: Parking
  18: Playground
  19: Pond
  20: Port
  21: RailwayStation
  22: Resort
  23: River
  24: School
  25: SparseResidential
  26: Square
  27: Stadium
  28: StorageTanks
  29: Viaduct


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [20]:
import torch.nn.functional as F

model_name = 'efficientnet_b0'
device = (
    'cuda' if torch.cuda.is_available() else
    'mps' if getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available() else
    'cpu'
)
print('Device:', device)

# 1) Load pretrained backbone
model = timm.create_model(model_name, pretrained=True)

# 2) Replace final classification layer for our dataset
model.reset_classifier(num_classes)
classifier = model.get_classifier()

# 3) Freeze backbone; train only classifier
for p in model.parameters():
    p.requires_grad = False
for p in classifier.parameters():
    p.requires_grad = True

model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(classifier.parameters(), lr=1e-3)

# -------------------------
# Clean training (on clean images)
# -------------------------
EPOCHS = 20
MAX_TRAIN_BATCHES = 500

for epoch in range(EPOCHS):
    model.train()
    running_loss = torch.tensor(0.0, device=device)
    correct = torch.tensor(0, device=device)
    total = 0
    t0 = time.time()
    for step, (x, y) in enumerate(train_loader):
        if step >= MAX_TRAIN_BATCHES:
            break
        x = x.to(device)
        y = y.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        # Avoid per-batch .item() (forces device sync on MPS)
        bs = x.size(0)
        running_loss += loss.detach() * bs
        correct += (logits.argmax(dim=1) == y).sum()
        total += y.numel()

    train_loss = (running_loss / max(1, total)).detach().cpu().item()
    train_acc = (correct.float() / max(1, total)).detach().cpu().item()
    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"train loss={train_loss:.4f} acc={train_acc:.4f} | "
        f"{time.time()-t0:.1f}s"
    )

# ---------------------------------------
# Evaluation-time corruptions (test only)
# ---------------------------------------
def add_gaussian_noise(x, sigma: float):
    return (x + torch.randn_like(x) * sigma).clamp(0.0, 1.0)

def motion_blur_horizontal(x, kernel_size: int):
    if kernel_size % 2 == 0:
        kernel_size += 1
    _, c, _, _ = x.shape
    weight = torch.ones((c, 1, 1, kernel_size), device=x.device, dtype=x.dtype) / kernel_size
    pad = kernel_size // 2
    x_pad = F.pad(x, (pad, pad, 0, 0), mode='reflect')
    return F.conv2d(x_pad, weight, groups=c)

def brightness_shift(x, factor: float):
    return (x * factor).clamp(0.0, 1.0)

@torch.inference_mode()
def evaluate(loader, corruption_fn=None):
    model.eval()
    correct = torch.tensor(0, device=device)
    total = 0
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        if corruption_fn is not None:
            x = corruption_fn(x)
        logits = model(x)
        correct += (logits.argmax(dim=1) == y).sum()
        total += y.numel()
    acc = (correct.float() / max(1, total)).detach().cpu().item()
    return acc

# Clean baseline accuracy
clean_acc = evaluate(test_loader)
print(f"Clean validation accuracy: {clean_acc:.4f}")

# Required corruption settings
gaussian_sigmas = [0.05, 0.1, 0.2]  # pixel-level Gaussian noise σ
MOTION_BLUR_KERNEL = 9              # motion blur (single setting)
BRIGHTNESS_FACTOR = 1.2             # brightness shift (single setting)

robustness_results = []

def record(corruption: str, level, acc_corrupted: float):
    robustness_results.append({
        'corruption': corruption,
        'level': level,
        'acc_corrupted': acc_corrupted,
        # Corruption Error = 1 - Acc_corrupted
        'corruption_error': 1.0 - acc_corrupted,
        # Relative Robustness = Acc_corrupted / Acc_clean
        'relative_robustness': (acc_corrupted / clean_acc) if clean_acc > 0 else float('nan'),
    })

# 1) Pixel-level Gaussian noise (σ = 0.05, 0.1, 0.2)
for sigma in gaussian_sigmas:
    acc = evaluate(test_loader, corruption_fn=lambda x, s=sigma: add_gaussian_noise(x, s))
    record('gaussian_noise', sigma, acc)

# 2) Motion blur
acc_blur = evaluate(test_loader, corruption_fn=lambda x: motion_blur_horizontal(x, MOTION_BLUR_KERNEL))
record('motion_blur', MOTION_BLUR_KERNEL, acc_blur)

# 3) Brightness shift
acc_bright = evaluate(test_loader, corruption_fn=lambda x: brightness_shift(x, BRIGHTNESS_FACTOR))
record('brightness_shift', BRIGHTNESS_FACTOR, acc_bright)

Device: cuda


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Epoch 1/20 | train loss=1.7115 acc=0.6371 | 53.8s
Epoch 2/20 | train loss=0.7821 acc=0.8214 | 52.9s
Epoch 3/20 | train loss=0.5962 acc=0.8495 | 53.3s
Epoch 4/20 | train loss=0.4957 acc=0.8751 | 53.8s
Epoch 5/20 | train loss=0.4255 acc=0.8940 | 51.7s
Epoch 6/20 | train loss=0.3690 acc=0.9085 | 54.3s
Epoch 7/20 | train loss=0.3424 acc=0.9140 | 53.7s
Epoch 8/20 | train loss=0.2994 acc=0.9252 | 51.5s
Epoch 9/20 | train loss=0.2769 acc=0.9320 | 53.5s
Epoch 10/20 | train loss=0.2439 acc=0.9439 | 53.4s
Epoch 11/20 | train loss=0.2279 acc=0.9447 | 51.4s
Epoch 12/20 | train loss=0.2148 acc=0.9495 | 53.3s
Epoch 13/20 | train loss=0.1973 acc=0.9557 | 51.7s
Epoch 14/20 | train loss=0.1923 acc=0.9549 | 52.8s
Epoch 15/20 | train loss=0.1760 acc=0.9590 | 53.0s
Epoch 16/20 | train loss=0.1651 acc=0.9636 | 51.4s
Epoch 17/20 | train loss=0.1552 acc=0.9655 | 52.9s
Epoch 18/20 | train loss=0.1464 acc=0.9679 | 51.1s
Epoch 19/20 | train loss=0.1463 acc=0.9660 | 53.2s
Epoch 20/20 | train loss=0.1390 acc=0.96

In [21]:
import pandas as pd
from IPython.display import Markdown, display

display(Markdown(
    "## Robustness report\n\n"
    f"**Clean validation accuracy:** `{clean_acc:.4f}`\n\n"
    "Required metrics:\n"
    "- **Corruption Error** $= 1 - \mathrm{Acc}_{\mathrm{corrupted}}$\n"
    "- **Relative Robustness** $= \mathrm{Acc}_{\mathrm{corrupted}}/\mathrm{Acc}_{\mathrm{clean}}$\n"
))

df = pd.DataFrame(robustness_results)
df = df[['corruption', 'level', 'acc_corrupted', 'corruption_error', 'relative_robustness']]
df = df.sort_values(['corruption', 'level']).reset_index(drop=True)
df.rename(columns={'acc_corrupted': 'val_acc'}, inplace=True)
df

<>:8: SyntaxWarning: invalid escape sequence '\m'
<>:9: SyntaxWarning: invalid escape sequence '\m'
<>:8: SyntaxWarning: invalid escape sequence '\m'
<>:9: SyntaxWarning: invalid escape sequence '\m'
/tmp/ipykernel_2480/3216939213.py:8: SyntaxWarning: invalid escape sequence '\m'
  "- **Corruption Error** $= 1 - \mathrm{Acc}_{\mathrm{corrupted}}$\n"
/tmp/ipykernel_2480/3216939213.py:9: SyntaxWarning: invalid escape sequence '\m'
  "- **Relative Robustness** $= \mathrm{Acc}_{\mathrm{corrupted}}/\mathrm{Acc}_{\mathrm{clean}}$\n"


## Robustness report

**Clean validation accuracy:** `0.8412`

Required metrics:
- **Corruption Error** $= 1 - \mathrm{Acc}_{\mathrm{corrupted}}$
- **Relative Robustness** $= \mathrm{Acc}_{\mathrm{corrupted}}/\mathrm{Acc}_{\mathrm{clean}}$


,corruption,level,val_acc,corruption_error,relative_robustness
0,brightness_shift,1.20,0.114449,0.885551,0.136054
1,gaussian_noise,0.05,0.144492,0.855508,0.171769
2,gaussian_noise,0.10,0.150215,0.849785,0.178571
3,gaussian_noise,0.20,0.097282,0.902718,0.115646
4,motion_blur,9.00,0.390558,0.609442,0.464286
